# Short-Term vs Long-Term Memory

Where short-term context-window memory ends and long-term cross-session memory begins, built on LangChain + LangGraph.

We will focus on one distinction: **short-term memory** is the context window the agent currently sees (finite, expensive, lost when the run ends), while **long-term memory** is durable knowledge that survives across sessions, threads, and process restarts. We will implement short-term memory end to end and feel exactly where it breaks long-term memory.

## Setup

Let's wire the model-agnostic stack and define the small helpers every section reuses.

Now, we will
- define a tiny `approx_tokens` helper so we can watch the context window grow,
- print a sanity-check confirming the LLM is ready.

In [1]:
# !pip install -q langchain langchain-google-genai langchain-openai langchain-anthropic langgraph langchain-community python-dotenv

from langchain.chat_models import init_chat_model
from langchain.embeddings import init_embeddings
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage, RemoveMessage, trim_messages
from langgraph.graph import StateGraph, START, END, MessagesState
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import InMemorySaver
from typing import TypedDict, List, Annotated
import os

from dotenv import load_dotenv
load_dotenv()

True

In [2]:
# os.environ['GEMINI_API_KEY']  # the variable for API key

llm   = init_chat_model('gpt-4o-mini', model_provider='openai', temperature=0)
embed = init_embeddings('sentence-transformers/all-MiniLM-L6-v2', provider='huggingface')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Now a tiny token estimator. We use word-count times `1.3` as a cheap proxy so we can *see* the context window grow without dragging in the model-specific tokenizer. The order of magnitude is what matters in this notebook.

In [3]:
def approx_tokens(messages) -> int:
    """Cheap token estimate. Word-count * 1.3 is within ~20% of real BPE counts for English chat,
    which is enough to watch growth trends. For production, swap in the model's real tokenizer."""
    words = sum(len(getattr(m, 'content', '').split()) for m in messages)
    return int(words * 1.3)

demo_msgs = [HumanMessage(content='hello there'), AIMessage(content='hi, how can we help today?')]
print('Setup OK | approx_tokens(demo) =', approx_tokens(demo_msgs))

Setup OK | approx_tokens(demo) = 10


## Naive list-buffer short-term memory

Let's start with the simplest possible short-term memory: a Python `list` we keep appending turns to, then prepend on the next prompt.

Now, we will
- define a global `BUFFER` list and a `buffered_chat(user_text)` function that appends every turn,
- run a 3-turn conversation where turn 3 references information from turn 1,
- print the resulting buffer so the mechanism is fully visible,
- name what we have built: a manual, framework-free **short-term** memory.

In [4]:
# Pattern: keep the whole transcript in a list; replay it on every call.
# This IS short-term memory in its rawest form -- no framework, no checkpointer.
BUFFER: List = []

def buffered_chat(user_text: str) -> str:
    """Append user turn, replay the whole BUFFER into the LLM, append the AI reply.
    The BUFFER list IS the short-term memory; nothing else persists state."""
    BUFFER.append(HumanMessage(content=user_text))
    reply = llm.invoke(BUFFER)
    BUFFER.append(reply)
    return reply.content

Now let's exercise it. Three turns: introduce a fact, add an unrelated turn, then ask the model to recall the fact. If short-term memory works, turn 3 should answer correctly because turns 1 and 2 are still in `BUFFER`.

In [5]:
BUFFER.clear()
print('=== Naive list-buffer short-term memory ===')
print('Turn 1 reply:', buffered_chat('My name is Priya and I work on agent systems.')[:120])
print('Turn 2 reply:', buffered_chat('I also have a dog named Milo.')[:120])
print('Turn 3 reply:', buffered_chat('What is my name and what do I work on?')[:160])
print(f'Buffer now holds {len(BUFFER)} messages (3 user + 3 AI = 6).')

=== Naive list-buffer short-term memory ===
Turn 1 reply: Nice to meet you, Priya! Agent systems are a fascinating area of study. What specific aspects of agent systems do you wo
Turn 2 reply: That’s great! Dogs can be wonderful companions. What breed is Milo, and does he have any favorite activities or quirks?
Turn 3 reply: Your name is Priya, and you work on agent systems.
Buffer now holds 6 messages (3 user + 3 AI = 6).


## `MessagesState` + `InMemorySaver` checkpointer

Now let's do the same thing the LangGraph way. The buffer becomes `MessagesState`, the global list becomes a checkpointed graph, and the conversation handle becomes a `thread_id`.

Now, we will
- define a single-node graph over `MessagesState` whose node calls the LLM on the whole message history,
- compile it with `checkpointer=InMemorySaver()` so state survives across `.invoke()` calls,
- invoke it 3 times on the same `thread_id` and confirm the conversation continues,
- invoke it once on a NEW `thread_id` and confirm short-term memory is per-thread.

In [6]:
def chat_node(state: MessagesState) -> dict:
    """Invoke the LLM on the full message history and return the reply.
    The returned dict is merged into state by the messages reducer, which APPENDS rather than replaces."""
    reply = llm.invoke(state['messages'])
    return {'messages': [reply]}

The graph topology is the smallest possible, START -> chat -> END. The teaching variable is the **checkpointer**, not the topology. Compiling with `InMemorySaver()` is what unlocks short-term memory across calls.

In [7]:
checkpointer = InMemorySaver()

builder = StateGraph(MessagesState)
builder.add_node('chat', chat_node)
builder.add_edge(START, 'chat')
builder.add_edge('chat', END)
graph = builder.compile(checkpointer=checkpointer)

print('Compiled graph nodes:', list(builder.nodes.keys()))

Compiled graph nodes: ['chat']


Now let's invoke it three times on the SAME `thread_id`. Same thread_id across invocations means *continue the same conversation*. Different `thread_id` means *fresh conversation, even though the process is the same*.

In [8]:
config_a = {'configurable': {'thread_id': 'demo-thread-a'}}

print('=== Same thread_id: short-term memory survives across .invoke() ===')
out1 = graph.invoke({'messages': [HumanMessage(content='My name is Priya.')]}, config=config_a)
print(f'---- thread demo-thread-a, turn 1 ----')
print(f'  AI: {out1["messages"][-1].content[:120]}')

out2 = graph.invoke({'messages': [HumanMessage(content='I work on agent systems.')]}, config=config_a)
print(f'---- thread demo-thread-a, turn 2 ----')
print(f'  AI: {out2["messages"][-1].content[:120]}')

out3 = graph.invoke({'messages': [HumanMessage(content='What is my name and what do I work on?')]}, config=config_a)
print(f'---- thread demo-thread-a, turn 3 ----')
print(f'  AI: {out3["messages"][-1].content[:160]}')

snap = graph.get_state(config_a)
print(f'Checkpoint for demo-thread-a now holds {len(snap.values["messages"])} messages.')

=== Same thread_id: short-term memory survives across .invoke() ===
---- thread demo-thread-a, turn 1 ----
  AI: Nice to meet you, Priya! How can I assist you today?
---- thread demo-thread-a, turn 2 ----
  AI: That sounds interesting! Agent systems can encompass a wide range of topics, from artificial intelligence and robotics t
---- thread demo-thread-a, turn 3 ----
  AI: Your name is Priya, and you work on agent systems. If there's anything specific you'd like to discuss or ask about agent systems, feel free to let me know!
Checkpoint for demo-thread-a now holds 6 messages.


*Tip: to clear a thread, just start a new `thread_id`. There is no built-in delete in `InMemorySaver`.*

In [9]:
config_b = {'configurable': {'thread_id': 'demo-thread-b'}}

print('---- thread demo-thread-b (FRESH thread_id) ----')
out_b = graph.invoke({'messages': [HumanMessage(content='What is my name?')]}, config=config_b)
print(f'  AI: {out_b["messages"][-1].content[:160]}')
print('Short-term memory is per-thread: a NEW thread_id starts from zero history.')

---- thread demo-thread-b (FRESH thread_id) ----
  AI: I'm sorry, but I don't have access to personal information about you unless you share it with me. How can I assist you today?
Short-term memory is per-thread: a NEW thread_id starts from zero history.


## Sliding window truncation

Now let's see what happens when the conversation grows past what we want to send to the LLM. The cheapest mitigation is a **sliding window**; keep only the last N messages.

Now, we will
- define `windowed_chat_node` that calls `trim_messages(strategy='last', ...)` BEFORE invoking the LLM,
- compile a fresh graph with this node and a new `InMemorySaver`,
- seed the conversation with the user's name, then bury it under several filler turns,
- ask about the buried fact and watch the agent fail because it is now outside the **sliding window**.

In [10]:
MAX_WINDOW_TOKENS = 40  # small on purpose so the window effect shows in a short demo

def windowed_chat_node(state: MessagesState) -> dict:
    """Trim the message history to the most recent MAX_WINDOW_TOKENS words BEFORE calling the LLM.
    The full history stays in the checkpoint (for audit); only the window is what the LLM sees."""
    trimmed = trim_messages(
        state['messages'],
        strategy='last',
        max_tokens=MAX_WINDOW_TOKENS,
        token_counter=len,
        include_system=True,
        allow_partial=False,
        start_on='human',
    )
    reply = llm.invoke(trimmed)
    return {'messages': [reply]}

We keep the system prompt always (`include_system=True`) and the *tail* of the conversation (`strategy='last'`), agent identity should never be evicted by a window.

In [11]:
builder = StateGraph(MessagesState)
builder.add_node('chat', windowed_chat_node)
builder.add_edge(START, 'chat')
builder.add_edge('chat', END)
window_graph = builder.compile(checkpointer=InMemorySaver())

window_config = {'configurable': {'thread_id': 'demo-window'}}

FILLER = [
    'I live in Bengaluru.',
    'My favourite language is Python.',
    'My team ships on Tuesdays.',
    'I drink south-indian filter coffee.',
    'My manager is Arjun.',
    'I have a dog named Milo.',
    'I am learning LangGraph.',
    'We use a 5-person squad.',
]

# Seed the buried fact, then push it out of the sliding window with filler.
window_graph.invoke({'messages': [HumanMessage(content='My name is Priya.')]}, config=window_config)
for f in FILLER:
    window_graph.invoke({'messages': [HumanMessage(content=f)]}, config=window_config)

out = window_graph.invoke({'messages': [HumanMessage(content='What is my name?')]}, config=window_config)
print('=== Sliding window: ask about a fact buried beyond the window ===')
print(f'  AI: {out["messages"][-1].content[:200]}')

full = window_graph.get_state(window_config).values['messages']
view = trim_messages(full, strategy='last', max_tokens=MAX_WINDOW_TOKENS,
                     token_counter=len, include_system=True,
                     allow_partial=False, start_on='human')
print(f'Checkpoint holds {len(full)} messages; LLM only sees {len(view)} after trimming.')
for m in view:
    print(f'  {m.__class__.__name__}: {m.content[:60]}')

=== Sliding window: ask about a fact buried beyond the window ===
  AI: Your name is Priya!
Checkpoint holds 20 messages; LLM only sees 20 after trimming.
  HumanMessage: My name is Priya.
  AIMessage: Nice to meet you, Priya! How can I assist you today?
  HumanMessage: I live in Bengaluru.
  AIMessage: That's great! Bengaluru is known for its vibrant culture, te
  HumanMessage: My favourite language is Python.
  AIMessage: Python is a fantastic language! It's versatile and widely us
  HumanMessage: My team ships on Tuesdays.
  AIMessage: Shipping on Tuesdays sounds like a solid schedule! It gives 
  HumanMessage: I drink south-indian filter coffee.
  AIMessage: South Indian filter coffee is delicious! The rich aroma and 
  HumanMessage: My manager is Arjun.
  AIMessage: It's nice to know about your manager, Arjun! How is your exp
  HumanMessage: I have a dog named Milo.
  AIMessage: Milo sounds adorable! Dogs can bring so much joy and compani
  HumanMessage: I am learning LangGraph.
 

## Summary buffer compression

Let's preserve the gist of old turns instead of just dropping them. When the buffer exceeds K turns, we ask the LLM to summarise the oldest slice and replace those messages with a single `SystemMessage` carrying the **summary buffer**.

Now, we will
- define `SummaryState` carrying both `messages` and a running `summary` string,
- implement `summarising_chat_node` (prepends the summary as a SystemMessage on each call) and `compress_node` (compresses old messages with `RemoveMessage`),
- wire them through a conditional edge that compresses only when the buffer exceeds the threshold,
- run a long conversation and confirm an early fact survives via the summary even after the original message is gone.

In [12]:
class SummaryState(TypedDict):
    messages: Annotated[list, add_messages]
    summary:  str

SUMMARISE_THRESHOLD = 6   # trigger summarisation when history exceeds this
KEEP_RECENT         = 2   # always keep this many recent messages verbatim

Now the prompt and the two nodes. `SUMMARY_PROMPT` lives at module level as an UPPER_CASE constant and is used via `.format()` so it is easy to find and re-tune.

In [13]:
SUMMARY_PROMPT = """You maintain a running summary of a chat conversation.
Update the summary with the new messages below. Keep proper nouns,
preferences, and commitments. Be concise (<= 120 words).

PRIOR SUMMARY:
{prior}

NEW MESSAGES:
{old}
"""

def summarising_chat_node(state: SummaryState) -> dict:
    """Main chat step. If state carries a summary, prepend it as a SystemMessage so the LLM
    has compressed context even after old messages are deleted from the buffer."""
    msgs = list(state['messages'])
    summary = state.get('summary', '')
    if summary:
        msgs = [SystemMessage(content=f'Summary of earlier conversation: {summary}')] + msgs
    reply = llm.invoke(msgs)
    return {'messages': [reply]}

def compress_node(state: SummaryState) -> dict:
    """Compress everything except the last KEEP_RECENT messages into the running summary,
    then emit RemoveMessage(id=...) for each compressed message so the reducer drops them."""
    msgs = state['messages']
    if len(msgs) <= SUMMARISE_THRESHOLD:
        return {}
    old = msgs[:-KEEP_RECENT]
    old_text = '\n'.join(f"{m.__class__.__name__}: {m.content}" for m in old)
    prior = state.get('summary', '') or '(none)'
    new_summary = llm.invoke(SUMMARY_PROMPT.format(prior=prior, old=old_text)).content
    removals = [RemoveMessage(id=m.id) for m in old if getattr(m, 'id', None)]
    return {'summary': new_summary, 'messages': removals}

Now the routing function and the graph wiring. After every chat turn we decide whether to compress.

In [14]:
def route_after_chat(state: SummaryState) -> str:
    """Compress when the message buffer exceeds the threshold; otherwise end."""
    return 'compress' if len(state['messages']) > SUMMARISE_THRESHOLD else 'end'

builder = StateGraph(SummaryState)
builder.add_node('chat', summarising_chat_node)
builder.add_node('compress', compress_node)
builder.add_edge(START, 'chat')
builder.add_conditional_edges('chat', route_after_chat, {'compress': 'compress', 'end': END})
builder.add_edge('compress', END)
summary_graph = builder.compile(checkpointer=InMemorySaver())

summary_config = {'configurable': {'thread_id': 'demo-summary'}}

summary_graph.invoke(
    {'messages': [HumanMessage(content='My name is Priya and I live in Bengaluru.')]},
    config=summary_config,
)
for f in FILLER[:6]:
    summary_graph.invoke({'messages': [HumanMessage(content=f)]}, config=summary_config)

snap = summary_graph.get_state(summary_config).values
print('=== Summary buffer after compression ===')
print('Running summary:')
print(f'  {snap.get("summary", "(empty)")[:240]}')
print(f'Buffer now holds {len(snap["messages"])} verbatim messages (rest live in the summary).')

probe = summary_graph.invoke(
    {'messages': [HumanMessage(content='What is my name and where do I live?')]},
    config=summary_config,
)
print(f'  Probe AI: {probe["messages"][-1].content[:200]}')

=== Summary buffer after compression ===
Running summary:
  SUMMARY:  
Priya lives in Bengaluru and enjoys the city's vibrant culture, pleasant weather, beautiful parks, and diverse food scene. Her favorite programming language is Python, which she appreciates for its simplicity and versatility. She
Buffer now holds 2 verbatim messages (rest live in the summary).
  Probe AI: Your name is Priya, and you live in Bengaluru.


## Failure mode: context overflow

Now let's deliberately overflow the **context window** with the simplest agent (no trimming, no summary) so we can see and name the failure mode that motivates long-term memory.

Now, we will
- reuse the plain `chat_node` graph from earlier with a fresh `InMemorySaver`,
- seed the conversation with a critical fact, then drown it in many high-volume filler turns,
- watch the message buffer grow turn over turn and the approximate token count climb,
- finally ask about the seeded fact and read the agent either forgetting it or producing a confused recall.

In [15]:
builder = StateGraph(MessagesState)
builder.add_node('chat', chat_node)
builder.add_edge(START, 'chat')
builder.add_edge('chat', END)
overflow_graph = builder.compile(checkpointer=InMemorySaver())

overflow_config = {'configurable': {'thread_id': 'demo-overflow'}}

VOLUMINOUS = (
    'Please write a verbose paragraph about it. ' * 6
)

overflow_graph.invoke(
    {'messages': [HumanMessage(content='My employee id is E-204. Please remember it.')]},
    config=overflow_config,
)

for i, topic in enumerate(['Bengaluru weather', 'filter coffee', 'Tuesday releases',
                           'Python typing', 'agent loops', 'OKR planning'], start=1):
    overflow_graph.invoke(
        {'messages': [HumanMessage(content=f'Tell me about {topic}. {VOLUMINOUS}')]},
        config=overflow_config,
    )
    snap_msgs = overflow_graph.get_state(overflow_config).values['messages']
    print(f'---- overflow turn {i}: msgs={len(snap_msgs)}  approx_tokens={approx_tokens(snap_msgs)} ----')

---- overflow turn 1: msgs=4  approx_tokens=426 ----
---- overflow turn 2: msgs=6  approx_tokens=825 ----
---- overflow turn 3: msgs=8  approx_tokens=1179 ----
---- overflow turn 4: msgs=10  approx_tokens=1543 ----
---- overflow turn 5: msgs=12  approx_tokens=1931 ----
---- overflow turn 6: msgs=14  approx_tokens=2327 ----


Now the punchline. We ask about the seeded employee id after the buffer has bloated. With no trimming and no summary, the buffer keeps growing, eventually it either exceeds the model's **context window** outright or the early fact gets lost in the noise.

In [16]:
out = overflow_graph.invoke(
    {'messages': [HumanMessage(content='What was my employee id again?')]},
    config=overflow_config,
)
print('=== Probe after deliberate context bloat ===')
print(f'  AI: {out["messages"][-1].content[:240]}')

final_msgs = overflow_graph.get_state(overflow_config).values['messages']
print(f'Final buffer: {len(final_msgs)} messages, approx_tokens={approx_tokens(final_msgs)}.')
print('Short-term-only agents hit a wall: every turn costs more tokens, every restart wipes everything,')
print('and a new thread starts from zero. That is the gap LONG-TERM memory fills (next session).')

=== Probe after deliberate context bloat ===
  AI: I'm sorry, but I can't store or recall personal information like employee IDs. If you need assistance with something else, feel free to ask!
Final buffer: 16 messages, approx_tokens=2366.
Short-term-only agents hit a wall: every turn costs more tokens, every restart wipes everything,
and a new thread starts from zero. That is the gap LONG-TERM memory fills (next session).


### Adopting short-term memory in production
- Swap `InMemorySaver` for `SqliteSaver` (or `PostgresSaver`) before deploy. The interface is identical; loss of state on a pod restart is a development convenience, not a production property.
- Always derive `thread_id` from your auth layer, never from user-controllable input. Two users sharing a `thread_id` means leaked conversation.
- Combine trimming and summarisation in production: trim aggressively to the last few turns, summarise everything that gets trimmed. This gives recent detail AND long-horizon gist within one bounded prompt.
- Set `temperature=0` on the summariser. Stable summaries make the running buffer reproducible across runs.

### Common pitfalls in short-term memory
- Token-counter accuracy matters at the boundary. `len(text.split())` undercounts vs real tokenizers by ~30%; budget 50% under the model limit until you wire in a real tokenizer.
- `RemoveMessage` is the only built-in way to delete from `MessagesState` the reducer is append-only otherwise. <br>Mitigation: emit `RemoveMessage(id=...)` for each message you want gone, in the same node return that adds the new summary.
- Short-term memory is bounded by the **context window**, scoped to one thread, and ephemeral to one process. <br>Mitigation: when you need cross-thread or cross-restart facts, reach for long-term memory, not bigger windows.